# Common Voice — GENİŞ görülmemiş test (400 örnek)

Mevcut CV-görülmemiş testi sadece 80 örnek (~600 kelime) — %30 WER civarında beklenen
rastgele dalgalanma `sqrt(0.30 × 0.70 / 600) ≈ ±1.8 puan`. Deney 3 (%28.99) ile
Deney 4 (%31.40) arasındaki 2.4 puanlık fark bu yüzden **istatistiksel olarak anlamlı
değil** (~1.3 sigma).

Bu notebook test setini **400 örneğe** çıkarır (gürültü ~±0.8 puana düşer) ve belirtilen
checkpoint'leri bu geniş sette ölçer.

**Dışlananlar** (hiçbiri eğitimde görülmemiş olmalı, hücre 3 sızıntı kontrolü yapar):
- Deney 3'ün eğittiği 5000 CV (SEED=42, TÜİK ağırlıklı)
- Deney 4'ün eğittiği 2500 CV (SEED+100=142, TÜİK ağırlıklı)

Eski 80 örnek yeni sete **dahil edilir** (ikisi de o 80'i görmedi) — böylece yeni 400'lük
sonuç eski 80'lik sonucun genişletilmiş hali olur, karşılaştırılabilir.

**Önbellek**: Test seti (jsonl + WAV'lar) Drive'a yazılır, ikinci çalıştırmada 621 MB'lık
Common Voice havuzu tekrar çekilmez.

In [ ]:
# 1) Kurulum - torchao'ya HIC dokunmuyoruz (kurmuyoruz, kuruluysa kaldiriyoruz)
!pip install -q evaluate jiwer soundfile transformers peft accelerate datasets
!pip uninstall -y torchao
!apt-get -qq install -y libsndfile1 > /dev/null

In [ ]:
# 2) Ayarlar
import os, re, json, random, gc
from collections import defaultdict
from pathlib import Path
import torch
import torchaudio as ta
import soundfile as sf
from datasets import load_dataset
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import evaluate
from google.colab import drive

drive.mount('/content/drive')
os.environ["HF_HUB_DISABLE_XET"] = "1"

INPUT_PATH = "/content/drive/MyDrive/colab_aktarim"
HEDEF_SR = 16000

# Deney 3 ve Deney 4'un CV secimlerini BIREBIR yeniden uretmek icin gereken sabitler
SEED = 42
TEST_SEED = 43            # eski 80'lik testin tohumu
N_DENEY3_CV = 5000        # Deney 3'un egittigi CV miktari
N_ESKI_TEST = 80          # eski test seti boyutu
N_DENEY4_CV = 2500        # Deney 4'un egittigi CV miktari
GENIS_TEST_SEED = 4343    # yeni, taze tohum
N_GENIS_TEST = 400        # hedef test seti boyutu (eski 80 dahil)

# Olculecek modeller: (checkpoint yolu ya da None=referans, etiket)
OLCULECEKLER = [
    (None, "whisper-base (LoRA'siz, referans)"),
    (f"{INPUT_PATH}/checkpoints/ilac_genis_r16/checkpoint-7798", "Deney3 r16 5000CV ep14"),
    (f"{INPUT_PATH}/checkpoints/ilac_karma_r32/checkpoint-7798", "Deney4 r32 karma  ep14"),
    (f"{INPUT_PATH}/checkpoints/ilac_karma_r32/checkpoint-8355", "Deney4 r32 karma  ep15"),
]

ONBELLEK = f"{INPUT_PATH}/data/cv_genis_test_400.jsonl"
# Sesler DRIVE'a yazilir - /content runtime yeniden baslatilinca silinir ve onbellek
# ise yaramaz olurdu (yasandi). 400 WAV ~65 MB, Drive icin sorun degil.
SES_DIZIN = Path(f"{INPUT_PATH}/data/cv_genis_test_sesler")
os.makedirs(f"{INPUT_PATH}/data", exist_ok=True)

In [ ]:
# 3) Test setini belirle - Deney3'un 5000'i ve Deney4'un 2500'u DISLANARAK
CV_REPO = "ysdede/commonvoice_17_tr_fixed"

TUIK_YAS_DAGILIMI = {
    "teens": 0.093, "twenties": 0.187, "thirties": 0.183, "fourties": 0.184,
    "fifties": 0.145, "sixties": 0.115, "seventies": 0.067, "eighties": 0.026,
}
CINSIYETLER = ["female_feminine", "male_masculine"]


def tuik_sec(rnd, havuz, hedef_toplam):
    """Egitim notebook'larindaki inline mantikla BIREBIR ayni - dislama
    hesaplarinin bit-bit ayni sonuc vermesi buna bagli."""
    bucketlar = defaultdict(list)
    for r in havuz:
        bucketlar[(r["age"], r["gender"])].append(r)
    secilen = []
    for yas, yas_agirlik in TUIK_YAS_DAGILIMI.items():
        for cinsiyet in CINSIYETLER:
            hedef_sayi = round(hedef_toplam * yas_agirlik * 0.5)
            mevcut = bucketlar.get((yas, cinsiyet), [])
            secilen.extend(rnd.sample(mevcut, min(hedef_sayi, len(mevcut))))
    eksik = hedef_toplam - len(secilen)
    if eksik > 0:
        secili_id = {id(s) for s in secilen}
        kalanlar = [r for r in havuz if id(r) not in secili_id]
        secilen.extend(rnd.sample(kalanlar, min(eksik, len(kalanlar))))
    return secilen


def anahtarla(kayitlar):
    return {(r["kaynak"], r["idx"]) for r in kayitlar}


def haric(havuz, *anahtar_kumeleri):
    yasak = set().union(*anahtar_kumeleri) if anahtar_kumeleri else set()
    return [r for r in havuz if (r["kaynak"], r["idx"]) not in yasak]


onbellek_gecerli = False
if os.path.exists(ONBELLEK) and os.path.getsize(ONBELLEK) > 0:
    with open(ONBELLEK, encoding="utf-8") as f:
        test_satirlar = [json.loads(s) for s in f]
    eksik_ses = [s for s in test_satirlar if not os.path.exists(s["audio_path"])]
    if test_satirlar and not eksik_ses:
        onbellek_gecerli = True
        print(f"Onbellekten yuklendi: {len(test_satirlar)} ornek (CV havuzu tekrar cekilmiyor)")
    else:
        print(f"Onbellek gecersiz ({len(test_satirlar)} satir, {len(eksik_ses)} ses eksik) - yeniden uretilecek")

if not onbellek_gecerli:
    print("Common Voice (tr) meta-verisi cekiliyor (~621 MB, birkac dakika)...")
    cv_train = load_dataset("parquet", data_files={"train": f"hf://datasets/{CV_REPO}/data/train-00000-of-00001.parquet"})["train"]
    cv_validated = load_dataset("parquet", data_files={"validated": f"hf://datasets/{CV_REPO}/data/validated-00000-of-00001.parquet"})["validated"]
    print(f"train: {len(cv_train)}, validated: {len(cv_validated)}")

    # Sadece meta-veri kolonlari - ses YOK (RAM guvenligi)
    meta_train = [{"kaynak": "train", "idx": i, "age": r["age"], "gender": r["gender"]}
                  for i, r in enumerate(cv_train.select_columns(["age", "gender"]))]
    meta_validated = [{"kaynak": "validated", "idx": i, "age": r["age"], "gender": r["gender"]}
                      for i, r in enumerate(cv_validated.select_columns(["age", "gender"]))]
    etiketli_meta = [r for r in meta_train + meta_validated
                     if r["age"] in TUIK_YAS_DAGILIMI and r["gender"] in CINSIYETLER]
    print(f"TUIK kategorileriyle eslesen: {len(etiketli_meta)}")

    # 3a) Deney 3'un egittigi 5000
    d3 = anahtarla(tuik_sec(random.Random(SEED), etiketli_meta, N_DENEY3_CV))

    # 3b) Eski 80'lik CV testi (Deney 4 bunu da dislamisti)
    eski80_kayit = random.Random(TEST_SEED).sample(haric(etiketli_meta, d3), N_ESKI_TEST)
    eski80 = anahtarla(eski80_kayit)

    # 3c) Deney 4'un egittigi 2500
    d4 = anahtarla(tuik_sec(random.Random(SEED + 100), haric(etiketli_meta, d3, eski80), N_DENEY4_CV))

    print(f"Deney3 egitim: {len(d3)}, Deney4 egitim: {len(d4)}, eski test: {len(eski80)}")

    # 3d) Yeni genis test = eski 80 + ikisinin de gormedigi havuzdan taze 320
    taze_havuz = haric(etiketli_meta, d3, d4, eski80)
    print(f"Iki deneyin de gormedigi taze havuz: {len(taze_havuz)}")
    ek = random.Random(GENIS_TEST_SEED).sample(taze_havuz, N_GENIS_TEST - N_ESKI_TEST)
    genis_meta = eski80_kayit + ek
    print(f"Genis test seti: {len(genis_meta)} ornek (eski {N_ESKI_TEST} + taze {len(ek)})")

    # Sizinti kontrolu
    genis_anahtar = anahtarla(genis_meta)
    for ad, kume in (("Deney3 egitimi", d3), ("Deney4 egitimi", d4)):
        kesisim = genis_anahtar & kume
        print(f"  {ad} ile kesisim: {len(kesisim)}" + ("  <-- SORUN!" if kesisim else "  OK"))

    # 3e) Gercek veriyi (ses dahil) cek, Drive'a WAV yaz
    SES_DIZIN.mkdir(parents=True, exist_ok=True)
    tr_idx = sorted(m["idx"] for m in genis_meta if m["kaynak"] == "train")
    va_idx = sorted(m["idx"] for m in genis_meta if m["kaynak"] == "validated")
    ornekler = list(cv_train.select(tr_idx)) + list(cv_validated.select(va_idx))
    random.Random(GENIS_TEST_SEED).shuffle(ornekler)

    test_satirlar = []
    for i, ornek in enumerate(ornekler):
        dalga = torch.tensor(ornek["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        sr = ornek["audio"]["sampling_rate"]
        if sr != HEDEF_SR:
            dalga = ta.functional.resample(dalga, sr, HEDEF_SR)
        metin = ornek.get("transcription")
        if not metin or dalga.shape[-1] < HEDEF_SR * 0.5:
            continue
        yol = SES_DIZIN / f"cvgenis_{i:04d}.wav"
        sf.write(str(yol), dalga[0].numpy(), HEDEF_SR, subtype="PCM_16")
        test_satirlar.append({"audio_path": str(yol), "text": metin})
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(ornekler)} WAV yazildi")

    with open(ONBELLEK, "w", encoding="utf-8") as f:
        for s in test_satirlar:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"Onbellege yazildi: {ONBELLEK} ({len(test_satirlar)} ornek)")

    del cv_train, cv_validated
    gc.collect()

kelime_sayisi = sum(len(s["text"].split()) for s in test_satirlar)
print(f"\nTest seti: {len(test_satirlar)} ornek, {kelime_sayisi} kelime")
print(f"Beklenen gurultu (%30 WER'de): +-{100 * (0.3 * 0.7 / kelime_sayisi) ** 0.5:.2f} puan")

In [ ]:
# 4) Modelleri genis test setinde olc
wer_metrigi = evaluate.load("wer")


def normalize_et(m):
    m = m.lower()
    m = re.sub(r"[.,!?;:\"'()]", "", m)
    return re.sub(r"\s+", " ", m).strip()


processor = WhisperProcessor.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")


def degerlendir(checkpoint_yolu, etiket):
    base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")
    base_model.generation_config.language = "turkish"
    base_model.generation_config.task = "transcribe"
    if checkpoint_yolu is None:
        model = base_model
    else:
        model = PeftModel.from_pretrained(base_model, checkpoint_yolu).merge_and_unload()
    model = model.to("cuda").eval()

    pred_strs, label_strs = [], []
    for i, satir in enumerate(test_satirlar):
        dalga, sr = sf.read(satir["audio_path"], dtype="float32")
        ozellikler = processor.feature_extractor(dalga, sampling_rate=sr).input_features
        girdi = torch.tensor(ozellikler).to("cuda")
        with torch.no_grad():
            tahmin_ids = model.generate(girdi, language="turkish", task="transcribe", max_length=128)
        pred_strs.append(processor.tokenizer.decode(tahmin_ids[0], skip_special_tokens=True))
        label_strs.append(satir["text"])
        if (i + 1) % 100 == 0:
            print(f"    {i + 1}/{len(test_satirlar)}")

    ham = 100 * wer_metrigi.compute(predictions=pred_strs, references=label_strs)
    norm = 100 * wer_metrigi.compute(
        predictions=[normalize_et(p) for p in pred_strs],
        references=[normalize_et(l) for l in label_strs],
    )
    print(f"[{etiket}] ham_wer={ham:.2f} normalize_wer={norm:.2f}")
    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()
    return ham, norm


sonuclar = []
for yol, etiket in OLCULECEKLER:
    print(f"\nOlculuyor: {etiket}")
    if yol is not None and not os.path.isdir(yol):
        print(f"  ATLANDI - checkpoint Drive'da yok: {yol}")
        continue
    try:
        ham, norm = degerlendir(yol, etiket)
        sonuclar.append((etiket, ham, norm))
    except Exception as e:
        print(f"  ATLANDI: {type(e).__name__}: {e}")

print(f"\n\n=== OZET: CV gorulmemis GENIS test ({len(test_satirlar)} ornek, {kelime_sayisi} kelime) ===")
print(f"{'model':<34} {'ham':>8} {'normalize':>10}")
for etiket, ham, norm in sonuclar:
    print(f"{etiket:<34} {ham:>7.2f}% {norm:>9.2f}%")